In [ ]:
import pandas as pd
import sys
import os

if "Desktop" in os.getcwd():
    base_path = "."
else:
    base_path = "."

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import (
    Datasets,
    DATA_FOLDER_PREFIX,
    Dataset,
    NELA_SOURCE_LABELS,
)
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv, save_to_csv, read_model_feature_injected_csv
from src.weak_labeling.consts.globals import NLP

# covid_df: pd.DataFrame = read_raw_data(Datasets.NELA_COVID_2021_DATASET)
# elections_df: pd.DataFrame = read_raw_data(Datasets.NELA_ELECTIONS_DATASET_SAMPLED)

In [ ]:
from src.weak_labeling.consts.content_features import ContentFeatures
from src.weak_labeling.content_based.threshold_selection.consts import FEATURE_PERCENTILES
from src.preprocessing.cleanup import separate_real_and_fake


df = read_csv(Datasets.NELA_ELECTIONS_2020.injected_path)
real_df, fake_df = separate_real_and_fake(df)

real_desc = real_df.describe(percentiles=FEATURE_PERCENTILES)
fake_desc = fake_df.describe(percentiles=FEATURE_PERCENTILES)

In [ ]:
thresholds = {}

for feature in ContentFeatures.get_all():
    wps_desc_df = pd.concat(
        [real_desc[feature], fake_desc[feature]], axis=1, keys=["real", "fake"]
    )
    wps_desc_df["abs_diff"] = wps_desc_df["real"] - wps_desc_df["fake"]

    print(feature)
    print(wps_desc_df)

    percentiles_dict = {}
    for idx, row in wps_desc_df.iterrows():
        if "%" in row.name:
            percentiles_dict[int(str(row.name).split("%")[0]) / 100] = {
                "real": row.real,
                "fake": row.fake,
                "diff": row.real - row.fake,
            }

In [ ]:
thresholds = {}

feature = ContentFeatures.CONTENT_WORDS_PER_SENTENCE
wps_desc_df = pd.concat(
    [real_desc[feature], fake_desc[feature]], axis=1, keys=["real", "fake"]
)

print(feature)
print(wps_desc_df)

In [ ]:
wps_desc_df.to_csv("./src/custom_notebooks/elections_percentiles.csv", float_format='%.3f')